In [6]:
import uproot
import pandas as pd

chunks = []

with uproot.open("~/Scrivania/sampic_run51_merged.root") as file:
    tree = file["picoTree"]
    
    # Prima vediamo cosa c'è nel tree
    print("Branch disponibili:")
    print(tree.keys())
    print(f"\nNumero totale di eventi: {tree.num_entries}")

Branch disponibili:
['Detector', 'Feb', 'Multiplicity', 'Channel', 'Cell0TimeStamp', 'TimeInstant', 'TOTValue', 'PeakValue', 'Baseline', 'Waveform', 'Amplitude', 'ArraySize', 'xCoord', 'yCoord', 'Davide', 'Golia', '7PAD', 'MCP']

Numero totale di eventi: 638043


In [7]:
import uproot
import awkward as ak
import numpy as np
import pandas as pd

branch_list = [
    'Detector', 'Feb', 'Multiplicity', 'Channel',
    'Cell0TimeStamp', 'TimeInstant', 'TOTValue', 'PeakValue',
    'Baseline', 'Amplitude', 'ArraySize',
    'xCoord', 'yCoord', 'Davide', 'Golia', '7PAD', 'MCP'
]

branch_2d = [
    'Detector', 'Feb', 'Multiplicity', 'Channel',
    'Cell0TimeStamp', 'TimeInstant', 'TOTValue', 'PeakValue',
    'Baseline', 'Amplitude', 'xCoord', 'yCoord'
]

branch_1d = ['ArraySize', 'Davide', 'Golia', '7PAD', 'MCP']

chunks = []

with uproot.open("~/Scrivania/sampic_run51_merged.root") as file:
    tree = file["picoTree"]
    n_tot = tree.num_entries

    for i, chunk in enumerate(tree.iterate(branch_list, library="ak", step_size=50_000)):
        n_chunk = len(chunk)

        # ArraySize dice esattamente quante hit valide ci sono per evento
        array_size = ak.to_numpy(chunk['ArraySize'])  # shape (n_chunk,)

        # --- Branch 2D: taglia ogni riga a ArraySize invece di usare maschera -1 ---
        arr_2d = {k: ak.to_numpy(chunk[k]) for k in branch_2d}

        rows_2d = {k: [] for k in branch_2d}
        for evt in range(n_chunk):
            n_hit = array_size[evt]
            for k in branch_2d:
                rows_2d[k].append(arr_2d[k][evt, :n_hit])

        df_2d = pd.DataFrame({k: np.concatenate(rows_2d[k]) for k in branch_2d})

        # --- Branch 1D: ripeti per ogni hit dell'evento ---
        arr_1d = {k: ak.to_numpy(chunk[k]) for k in branch_1d}
        df_1d = pd.DataFrame({k: np.repeat(arr_1d[k], array_size) for k in branch_1d})

        # --- Multiplicity: estrai le 3 posizioni come colonne separate ---
        mult = ak.to_numpy(chunk['Multiplicity'])  # shape (n_chunk, 140)
        df_mult_per_evento = pd.DataFrame({
            'Mult_det0': mult[:, 0],
            'Mult_det1': mult[:, 1],
            'Mult_det2': mult[:, 2],
        })
        # Ripeti per ogni hit dell'evento
        df_mult = pd.DataFrame({
            col: np.repeat(df_mult_per_evento[col].values, array_size)
            for col in df_mult_per_evento.columns
        })

        # --- Indice evento ---
        event_idx = np.repeat(np.arange(i * 50_000, i * 50_000 + n_chunk), array_size)

        # --- Assembla ---
        df_chunk = pd.concat([
            pd.Series(event_idx, name='event_idx'),
            df_2d.reset_index(drop=True),
            df_1d.reset_index(drop=True),
            df_mult.reset_index(drop=True)
        ], axis=1)

        # Rimuovi Multiplicity grezza (era 2D, ora abbiamo le 3 colonne separate)
        df_chunk.drop(columns=['Multiplicity'], inplace=True)

        chunks.append(df_chunk)
        print(f"Chunk {i+1} — eventi: {n_chunk} — hit totali: {len(df_chunk)} — letti: {min((i+1)*50_000, n_tot)}/{n_tot}")

df = pd.concat(chunks, ignore_index=True)
print(f"\n✅ DataFrame finale: {df.shape}")
print(df.head(10))

Chunk 1 — eventi: 50000 — hit totali: 93499 — letti: 50000/638043
Chunk 2 — eventi: 50000 — hit totali: 72236 — letti: 100000/638043
Chunk 3 — eventi: 50000 — hit totali: 68193 — letti: 150000/638043
Chunk 4 — eventi: 50000 — hit totali: 75496 — letti: 200000/638043
Chunk 5 — eventi: 50000 — hit totali: 75299 — letti: 250000/638043
Chunk 6 — eventi: 50000 — hit totali: 66798 — letti: 300000/638043
Chunk 7 — eventi: 50000 — hit totali: 74427 — letti: 350000/638043
Chunk 8 — eventi: 50000 — hit totali: 67345 — letti: 400000/638043
Chunk 9 — eventi: 50000 — hit totali: 74660 — letti: 450000/638043
Chunk 10 — eventi: 50000 — hit totali: 74350 — letti: 500000/638043
Chunk 11 — eventi: 50000 — hit totali: 67146 — letti: 550000/638043
Chunk 12 — eventi: 50000 — hit totali: 67966 — letti: 600000/638043
Chunk 13 — eventi: 38043 — hit totali: 54899 — letti: 638043/638043

✅ DataFrame finale: (932314, 20)
   event_idx  Detector  Feb  Channel  Cell0TimeStamp   TimeInstant  TOTValue  \
0          0

In [8]:
import uproot
import awkward as ak
import numpy as np
import pandas as pd

# --- Parametri modificabili ---
X_TARGET = 5.0
Y_TARGET = 5.0

# Molteplicità per ogni detector — cambia solo questi!
MULT_DET0 = 1   # minimo hit detector 0
MULT_DET1 = 1   # minimo hit detector 1
MULT_DET2 = 0   # detector 2: whatever (0 = nessun vincolo)

# ------------------------------

branch_list = [
    'Detector', 'Feb', 'Channel', 'Cell0TimeStamp', 'TimeInstant',
    'TOTValue', 'PeakValue', 'Baseline', 'Amplitude', 'Waveform',
    'ArraySize', 'xCoord', 'yCoord', 'Multiplicity',
    'Davide', 'Golia', '7PAD', 'MCP'
]

branch_2d = [
    'Detector', 'Feb', 'Channel', 'Cell0TimeStamp', 'TimeInstant',
    'TOTValue', 'PeakValue', 'Baseline', 'Amplitude',
    'xCoord', 'yCoord'
]

branch_1d = ['ArraySize', 'Davide', 'Golia', '7PAD', 'MCP']

chunks = []

with uproot.open("~/Scrivania/sampic_run51_merged.root") as file:
    tree = file["picoTree"]
    n_tot = tree.num_entries

    for i, chunk in enumerate(tree.iterate(branch_list, library="ak", step_size=50_000)):
        n_chunk = len(chunk)

        array_size = ak.to_numpy(chunk['ArraySize'])     # (n_chunk,)
        mult       = ak.to_numpy(chunk['Multiplicity'])  # (n_chunk, 140)

        # xCoord e yCoord sono uguali per tutte le hit dell'evento, prendi la prima
        xcoord = ak.to_numpy(chunk['xCoord'])[:, 0]
        ycoord = ak.to_numpy(chunk['yCoord'])[:, 0]

        # --- Filtro evento ---
        mask_evento = (
            (xcoord == X_TARGET) &
            (ycoord == Y_TARGET) &
            (mult[:, 0] == MULT_DET0) &
            (mult[:, 1] == MULT_DET1) &
            (mult[:, 2] >= MULT_DET2)   # se MULT_DET2=0 è sempre True
        )

        if mask_evento.sum() == 0:
            continue

        n_sel = mask_evento.sum()
        array_size_filt = array_size[mask_evento]
        mult_filt       = mult[mask_evento]

        # --- Branch 2D ---
        arr_2d = {k: ak.to_numpy(chunk[k])[mask_evento] for k in branch_2d}
        rows_2d = {k: [] for k in arr_2d}
        for evt in range(n_sel):
            n_hit = array_size_filt[evt]
            for k in arr_2d:
                rows_2d[k].append(arr_2d[k][evt, :n_hit])
        df_2d = pd.DataFrame({k: np.concatenate(rows_2d[k]) for k in arr_2d})

        # --- Waveform ---
        waveform_raw = ak.to_numpy(chunk['Waveform'])[mask_evento]
        waveforms = []
        for evt in range(n_sel):
            n_hit = array_size_filt[evt]
            for hit in range(n_hit):
                waveforms.append(waveform_raw[evt, hit, :])
        df_2d['Waveform'] = waveforms

        # --- Branch 1D ---
        arr_1d = {k: ak.to_numpy(chunk[k])[mask_evento] for k in branch_1d}
        df_1d = pd.DataFrame({k: np.repeat(arr_1d[k], array_size_filt) for k in branch_1d})

        # --- Multiplicity come colonne separate ---
        df_mult = pd.DataFrame({
            'Mult_det0': np.repeat(mult_filt[:, 0], array_size_filt),
            'Mult_det1': np.repeat(mult_filt[:, 1], array_size_filt),
            'Mult_det2': np.repeat(mult_filt[:, 2], array_size_filt),
        })

        # --- Indice evento originale ---
        idx_originali = np.where(mask_evento)[0] + i * 50_000
        event_idx = np.repeat(idx_originali, array_size_filt)

        df_chunk = pd.concat([
            pd.Series(event_idx, name='event_idx'),
            df_2d.reset_index(drop=True),
            df_1d.reset_index(drop=True),
            df_mult.reset_index(drop=True)
        ], axis=1)

        chunks.append(df_chunk)
        print(f"Chunk {i+1} — eventi selezionati: {n_sel} — hit totali: {len(df_chunk)}")

df = pd.concat(chunks, ignore_index=True)
print(f"\n✅ DataFrame finale: {df.shape}")
print(df.head(5))

Chunk 2 — eventi selezionati: 54 — hit totali: 110
Chunk 3 — eventi selezionati: 60 — hit totali: 122
Chunk 4 — eventi selezionati: 96 — hit totali: 193
Chunk 5 — eventi selezionati: 81 — hit totali: 164
Chunk 6 — eventi selezionati: 56 — hit totali: 113
Chunk 7 — eventi selezionati: 74 — hit totali: 149
Chunk 8 — eventi selezionati: 59 — hit totali: 118
Chunk 9 — eventi selezionati: 82 — hit totali: 164
Chunk 10 — eventi selezionati: 75 — hit totali: 150
Chunk 11 — eventi selezionati: 65 — hit totali: 130
Chunk 12 — eventi selezionati: 56 — hit totali: 114
Chunk 13 — eventi selezionati: 65 — hit totali: 130

✅ DataFrame finale: (1657, 21)
   event_idx  Detector  Feb  Channel  Cell0TimeStamp   TimeInstant  TOTValue  \
0      77044       2.0    1       45    1.337558e+11  1.337558e+11       0.0   
1      77044       1.0    3       45    1.337558e+11  1.337558e+11       0.0   
2      77263       2.0    1       45    1.337586e+11  1.337586e+11       0.0   
3      77263       1.0    3     